In [11]:
# Install required packages (run once)
# %pip install openai tqdm python-dotenv
import os
import json
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI
import time

# Load environment variables
load_dotenv()

# --- CONFIGURATION ---
# Ensure "OPENROUTER_API_KEY" is in your .env file
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Selected Model: Gemini 2.0 Flash Experimental (Fastest & Smartest for this task)
MODEL_NAME = "google/gemini-2.0-flash-exp:free"

# Folder paths (Adjust these to match your actual local paths)
INPUT_DIR = Path("E:/graduation_project/data_json")
OUTPUT_DIR = Path("E:/graduation_project/json_llm_responses")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Initialize OpenRouter Client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"✅ Setup Complete. Target Model: {MODEL_NAME}")

✅ Setup Complete. Target Model: google/gemini-2.0-flash-exp:free


In [2]:
# List and select files
json_files = list(INPUT_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError("No JSON files found in input directory!")

print(f"Found {len(json_files)} JSON files:")
for i, f in enumerate(json_files):
    print(f"{i+1}. {f.name}")

# Simple input to pick a file
try:
    file_index = int(input("Enter file number: ")) - 1
    selected_file = json_files[file_index]
except (ValueError, IndexError):
    print("Invalid selection, defaulting to first file.")
    selected_file = json_files[0]

with open(selected_file, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"\n📂 Loaded {len(chunks)} chunks from: {selected_file.name}")

Found 1 JSON files:
1. Computer Systems A Programmers Perspective by Randal E. Bryant, David R. OHallaron (z-lib.org)_chunks.json

📂 Loaded 3315 chunks from: Computer Systems A Programmers Perspective by Randal E. Bryant, David R. OHallaron (z-lib.org)_chunks.json


In [ ]:
def process_chunk_with_llm(chunk_text, chunk_id):
    """
    Sends text to LLM. Handles Rate Limits (429) by waiting and retrying.
    """
    
    prompt = f"""
    You are an Expert Technical Editor for a Computer Science Textbook.
    
    ### INPUT TEXT (Raw OCR Extraction):
    "{chunk_text}"s

    ### TASK:
    1. **Analyze**: Is this content educational? 
       - DISCARD: Copyright pages, Table of Contents, Prefaces, ISBN lists, empty pages, or generic filler.
       - KEEP: Technical explanations, code snippets, diagrams descriptions, definitions, or summaries.
    
    2. **Edit & Polish** (If keeping):
       - You HAVE FREEDOM to add, remove, or reorder words to make the text clear, concise, and professional.
       - Fix broken sentences, merge fragments, and smooth out the flow.
       - **CRITICAL RULE**: Do NOT change the technical meaning or context. Ensure all facts remain accurate to the source.
    
    3. **Categorize**: Generate a specific 'Topic' (Chapter/Concept) and 'Subtopic' (Detail).
    
    ### OUTPUT FORMAT (Strict JSON Only):
    {{
        "source": "the refrence name and id of the chunk",
        "keep": boolean,
        "clean_text": "The polished, well-written version of the text (string)",
        "topic": "string (Main Concept)",
        "subtopic": "string (Specific Detail)"
    }}
    """
    
    retries = 0
    max_retries = 5
    wait_time = 10  # Start waiting 10 seconds

    while retries < max_retries:
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You are a JSON-only API. Output ONLY valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"}, 
                temperature=0.2 
            )
            
            response_content = completion.choices[0].message.content
            data = json.loads(response_content)
            
            # Handle list vs object return
            if isinstance(data, list):
                return data[0] if data else None
            return data

        except Exception as e:
            error_msg = str(e)
            
            # Check if it's a Rate Limit error (429)
            if "429" in error_msg or "rate limit" in error_msg.lower():
                print(f"⏳ Rate Limit hit on Chunk {chunk_id}. Waiting {wait_time}s...")
                time.sleep(wait_time)
                retries += 1
                wait_time *= 2  # Double the wait time for next try (10s -> 20s -> 40s...)
            else:
                # If it's another error (like context too long), just fail
                print(f"⚠️ Critical Error on Chunk {chunk_id}: {e}")
                return None

    print(f"❌ Failed Chunk {chunk_id} after {max_retries} retries.")
    return None

In [13]:
enriched_chunks = []
garbage_count = 0
error_count = 0

print(f"🚀 Starting robust processing with {MODEL_NAME}...")

# Processing loop
for i, chunk in enumerate(tqdm(chunks[6:20], desc="Enriching Chunks")):
    
    # 1. Skip tiny chunks
    if len(chunk.get("text", "")) < 50:
        garbage_count += 1
        continue
        
    # 2. Call LLM (With Retry Logic)
    result = process_chunk_with_llm(chunk["text"], chunk["id"])
    
    if result:
        if result.get("keep") is True:
            enriched_chunks.append({
                "id": chunk["id"],
                "source": chunk["source"],
                "text": result.get("clean_text", chunk["text"]),
                "topic": result.get("topic", "General"),
                "subtopic": result.get("subtopic", ""),
                "embedding_text": f"{result.get('topic')} - {result.get('subtopic')}: {result.get('clean_text')}"
            })
        else:
            garbage_count += 1
    else:
        error_count += 1

    # --- SAFETY SAVE EVERY 50 CHUNKS ---
    if (i + 1) % 50 == 0:
        temp_filename = OUTPUT_DIR / f"{selected_file.stem}_checkpoint.json"
        with open(temp_filename, "w", encoding="utf-8") as f:
            json.dump(enriched_chunks, f, ensure_ascii=False, indent=2)

print("\n" + "="*30)
print(f"✅ Final Processing Complete!")
print(f"Original Chunks: {len(chunks)}")
print(f"Useful Kept: {len(enriched_chunks)}")
print(f"Garbage Discarded: {garbage_count}")
print(f"Failed (Errors): {error_count}")

🚀 Starting robust processing with google/gemini-2.0-flash-exp:free...


Enriching Chunks:   7%|▋         | 1/14 [00:04<01:02,  4.79s/it]

⚠️ Error processing Chunk 7: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemini-2.0-flash-exp:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google'}}, 'user_id': 'user_340nYGGQR6AleRV9MLwPbHFw93r'}


Enriching Chunks:  14%|█▍        | 2/14 [00:07<00:43,  3.63s/it]

⚠️ Error processing Chunk 8: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1767398400000'}, 'provider_name': None}}, 'user_id': 'user_340nYGGQR6AleRV9MLwPbHFw93r'}


...

⚠️ Error processing Chunk 18: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '16', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1767365520000'}, 'provider_name': None}}, 'user_id': 'user_340nYGGQR6AleRV9MLwPbHFw93r'}


Enriching Chunks:  93%|█████████▎| 13/14 [00:27<00:01,  1.74s/it]

⚠️ Error processing Chunk 19: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '16', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1767365520000'}, 'provider_name': None}}, 'user_id': 'user_340nYGGQR6AleRV9MLwPbHFw93r'}


Enriching Chunks: 100%|██████████| 14/14 [00:29<00:00,  2.08s/it]

⚠️ Error processing Chunk 20: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '16', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1767365520000'}, 'provider_name': None}}, 'user_id': 'user_340nYGGQR6AleRV9MLwPbHFw93r'}

✅ Final Processing Complete!
Original Chunks: 3315
Useful Kept: 0
Garbage Discarded: 0
Failed (Errors): 14


In [9]:
output_filename = OUTPUT_DIR / f"{selected_file.stem}_enriched.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(enriched_chunks, f, ensure_ascii=False, indent=2)

print(f"💾 Saved enriched dataset to:\n{output_filename}")

# Preview first 3 kept chunks
print("\n--- Preview ---")
for c in enriched_chunks[:3]:
    print(f"ID: {c['id']}")
    print(f"Topic: {c['topic']} > {c['subtopic']}")
    print(f"Text Snippet: {c['text'][:100]}...")
    print("-" * 20)

💾 Saved enriched dataset to:
E:\graduation_project\json_llm_responses\Computer Systems A Programmers Perspective by Randal E. Bryant, David R. OHallaron (z-lib.org)_chunks_enriched.json

--- Preview ---
ID: 1
Topic: Book Information > Authors and Publication
Text Snippet: Computer Systems: A Programmer's Perspective, Third Edition, is authored by Randal E. Bryant of Carn...
--------------------
ID: 3
Topic: Publication Information > Copyright and Production Details
Text Snippet: This book was composed by Windfall Software and printed/bound by Courier Westford, with cover printi...
--------------------
ID: 4
Topic: Legal and Disclaimers > Trademark Usage and Liability
Text Snippet: Manufacturers often claim trademarks for their product designations. In this book, where the publish...
--------------------
